In [ ]:
!pip install -q langchain langchain-groq

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

In [ ]:
SYSTEM = """ROLE: You are the assistant for TastyBites Restaurant.

FACTS: We serve vegetarian and non-vegetarian food.
       Sample menu and prices:
       - Veg Biryani: Rs. 180
       - Paneer Butter Masala: Rs. 220
       - Chicken Biryani: Rs. 240
       - Masala Dosa: Rs. 80
       - Veg Fried Rice: Rs. 150
       - Chicken Noodles: Rs. 190
       - Gulab Jamun: Rs. 60
       - Fresh Lime Soda: Rs. 50
       Dine-in and takeaway are available.
       Live stock and delivery availability are not connected.

RULES: Recommend dishes based on customer preferences.
       Never invent menu items, prices, stock, or delivery availability.
       Ask questions if an order or food preference is unclear.
       Ask about allergies when relevant and never claim a dish is allergen-free
       unless confirmed by the restaurant.
       Do not claim that an order, payment, or reservation has been placed.
       If you don't know, say "I'll check with the restaurant."

FORMAT: Reply politely in under 60 words.
        Use friendly English or Telugu matching the customer."""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

In [ ]:
history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

In [ ]:
print(ask("Hi, I'm Ravi. Can you recommend a vegetarian dish?"))

print(ask("What is my name?"))

In [ ]:
history

In [ ]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    with open(path, newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()
        passed += int(ok)

        print(
            "PASS" if ok else "FAIL",
            "|", row["question"],
            "→", answer[:100]
        )

    total = len(rows)
    rate = passed / total if total else 0
    print(f"\n{passed}/{total} passed ({rate:.0%})")

In [ ]:
import pandas as pd

data = {
    "question": [
        "What is the price of Veg Biryani?",
        "What is the price of Masala Dosa?",
        "Do you have vegetarian food?",
        "Can you confirm that my order has been placed?"
    ],
    "must_include": [
        "180",
        "80",
        "vegetarian",
        "cannot"
    ]
}

df = pd.DataFrame(data)
df.to_csv("/content/test.csv", index=False)
print(df)

In [ ]:
run_tests()

In [ ]:
from langchain.agents import create_agent

agent_order = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result_order = agent_order.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Can you suggest a vegetarian meal for lunch?"
        }
    ]
})

print(result_order["messages"][-1].content)

In [ ]:
print(ask("What is the price of Chicken Biryani?"))

print(ask("I prefer spicy food. What do you recommend?"))

print(ask("What is my name?"))

print(ask("Can you guarantee delivery in 20 minutes?"))